# 02 — Trio Assembly

Demonstrates the `TrioLoader` pipeline: loading proband + father + mother variant
files, joining on `Gene + HGVS_Coding` to infer inheritance modes, and tagging
variants with phenotype data.

**Covers:**
1. Load and preview `trios.tsv` and `phenotypes.tsv`
2. Run `TrioLoader.load_all()` → assembled DataFrame
3. Inspect shape, columns, and sample rows
4. Inheritance-mode distribution across all trios
5. Per-trio variant counts and family-member overlap
6. Phenotype summary (HPO terms, affected status)
7. Save assembled DataFrame for downstream notebooks

## 1. Imports & Configuration

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

DATA_DIR       = ROOT / "data"
TRIOS_TSV      = DATA_DIR / "trios.tsv"
PHENOTYPES_TSV = DATA_DIR / "phenotypes.tsv"
OUT_PARQUET    = DATA_DIR / "trio_variants.parquet"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print("ROOT:", ROOT)

## 2. Preview `trios.tsv` and `phenotypes.tsv`

In [ ]:
trios = pd.read_csv(TRIOS_TSV, sep="\t")
phenos = pd.read_csv(PHENOTYPES_TSV, sep="\t")

print(f"Trios: {len(trios)} rows")
display(trios.head(5))

print(f"\nPhenotypes: {len(phenos)} rows  ({phenos['affected'].sum()} affected)")
display(phenos.head(8))

## 3. Run `TrioLoader.load_all()`

`TrioLoader` joins proband variants with father/mother on `Gene + HGVS_Coding`
to infer the inheritance mode for every variant row.

In [ ]:
from src.trio_loader import TrioLoader

loader = TrioLoader(
    trios_path=str(TRIOS_TSV),
    phenotypes_path=str(PHENOTYPES_TSV),
    data_dir=str(DATA_DIR),
)
df = loader.load_all()

print(f"Assembled DataFrame: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumn names:")
print(df.columns.tolist())
df.head(3)

## 4. Inheritance-Mode Distribution

In [ ]:
inh_counts = df["inheritance_mode"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

inh_counts.plot.bar(ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Inheritance mode — all variants")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=30)

# Same data broken down by pathogenicity label
pat_mask = df["Classification"].str.contains(
    "Pathogenic|Likely pathogenic", case=False, na=False
)
cross = pd.crosstab(df["inheritance_mode"], pat_mask.map({True: "Pathogenic", False: "Other"}))
cross.plot.bar(ax=axes[1], stacked=True, colormap="Set2", edgecolor="white")
axes[1].set_title("Inheritance mode by pathogenicity")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(title="Label", fontsize=8)

plt.tight_layout()
plt.show()

print("\nInheritance mode counts:")
print(inh_counts.to_string())

## 5. Per-Trio Variant Counts & Family-Member Overlap

In [ ]:
trio_summary = df.groupby("trio_id").agg(
    total_variants    = ("Gene", "count"),
    father_shared     = ("father_has_variant", "sum"),
    mother_shared     = ("mother_has_variant", "sum"),
    pathogenic_count  = ("Classification",
                         lambda s: s.str.contains("Pathogenic", case=False, na=False).sum()),
).reset_index()

display(trio_summary)

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(trio_summary))
width = 0.28
ax.bar([i - width for i in x], trio_summary["total_variants"],  width, label="Total",         color="steelblue")
ax.bar([i         for i in x], trio_summary["father_shared"],   width, label="Father shared",  color="coral")
ax.bar([i + width for i in x], trio_summary["mother_shared"],   width, label="Mother shared",  color="mediumseagreen")
ax.set_xticks(list(x))
ax.set_xticklabels(trio_summary["trio_id"], rotation=45, ha="right")
ax.set_ylabel("Variant count")
ax.set_title("Variant counts per trio")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Phenotype Summary

In [ ]:
# HPO term frequency across probands
proband_phenos = phenos[phenos["sample_id"].str.endswith("_proband")]
hpo_counts = (
    proband_phenos["hpo_terms"]
    .dropna()
    .str.split(";").explode()
    .str.strip()
    .value_counts()
)

fig, ax = plt.subplots(figsize=(8, 4))
hpo_counts.plot.bar(ax=ax, color="darkorange", edgecolor="white")
ax.set_title("HPO term frequency in probands")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print(f"\nProband affected rate: {proband_phenos['affected'].mean():.0%}")
print(f"All sample affected rate: {phenos['affected'].mean():.0%}")

## 7. Save Assembled DataFrame

Save the merged trio DataFrame as a Parquet file for use in `03_feature_engineering.ipynb`.

In [ ]:
df.to_parquet(OUT_PARQUET, index=False)
print(f"Saved → {OUT_PARQUET}  ({df.shape[0]:,} rows)")

# Quick sanity-check
reload = pd.read_parquet(OUT_PARQUET)
assert reload.shape == df.shape, "Shape mismatch on reload!"
print("Reload verified ✓")